# Police Station Spatial Intelligence — Isovist

## //00 Setup | Import Libraries
> All topologicpy modules needed for geometry, topology, and graph analysis.

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## //01 Version Check
> Confirm topologicpy meets the minimum required version (0.9.31+).

In [ ]:
print(Helper.Version())

## //02 Renderer | Configuration
> Set render target. Options: `vscode` | `colab` | `browser`.

In [ ]:
renderer = "vscode"

## //03 Utility Functions
> `reset_dictionaries` — clear face metadata | `transfer_dicts_by_key` — propagate graph values back to geometry.

In [ ]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)


## //04 Import Floor Plan
> Load the processed ground-floor BREP face from `gf-floor-plan_face.brep`.

In [ ]:
police_station = Topology.ByBREPPath(r"../assets/gf-floor-plan_face.brep")

## //05 Visualize Geometry
> Raw floor plan — single face, no grid.

In [ ]:
Topology.Show(police_station,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## //06 Grid Overlay
> Two grids: dense vertex grid for isovist viewpoints | coarse edge grid for shell slicing.

In [ ]:
b_r = Wire.BoundingRectangle(police_station)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange1 = list(range(0,int(width)+5,5))
vRange1 = list(range(0,int(length)+5,5))

uRange2 = list(range(0,int(width)+2,2))
vRange2 = list(range(0,int(length)+2,2))
grid1 = Grid.VerticesByDistances(police_station, clip=True, uRange=uRange1, vRange=vRange1)
grid2 = Grid.EdgesByDistances(police_station, clip=True, uRange=uRange2, vRange=vRange2)

## //07 Slice Floor Plan | Shell
> Divide the face using the edge grid. Each cell becomes a graph node.

In [ ]:
shell = Topology.Slice(police_station, grid2)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

## //08 Derive Analysis Graph

In [ ]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
analysis_graph = Graph.ByTopology(shell)

## //09 Store Vertices | Graph & Isovist Grid

In [ ]:
g_verts = Graph.Vertices(analysis_graph)
iso_verts = Topology.Vertices(grid1)

## //10 Spatial Analysis | Isovist Visibility

### //10a Create Isovists
> Compute one isovist face per grid viewpoint (~20 min). Errors during computation can be ignored.

In [ ]:
from tqdm import tqdm
import time

isovists = []
for v in tqdm(iso_verts, desc="Computing isovists"):
    isovist = Face.Isovist(police_station, v)
    isovists.append(isovist)

In [ ]:
import pickle

CACHE_PATH = r"../assets/isovists_cache3.pkl"

brep_strings = []
for iso in isovists:
    if iso is not None:
        brep_strings.append(Topology.BREPString(iso))
    else:
        brep_strings.append(None)

with open(CACHE_PATH, "wb") as f:
    pickle.dump(brep_strings, f)

print(f"Cached {sum(1 for b in brep_strings if b)} / {len(brep_strings)} isovists")

In [ ]:
Topology.Show(police_station, isovists,
              faceColorKey="cp_color",
              faceOpacity=0.6,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### //10b Compute Visibility Scores
> Count how many dense-grid vertices fall within each isovist's face.

In [ ]:
new_verts = []
n_list = []
for i, iso in enumerate(isovists):
    if iso is None:
        continue
    
    # Convert Cluster -> Face if needed
    if Topology.TypeAsString(iso) == "Cluster":
        iso_faces = Cluster.Faces(iso)
        if not iso_faces:
            continue
        # Take the face containing the viewpoint, or merge them
        iso = iso_faces[0]  # if there are multiple, this picks one; usually fine
    
    if Topology.TypeAsString(iso) != "Face":
        continue  # skip anything still not a Face
    
    v = iso_verts[i]
    b_list = Vertex.IsInternal2D(g_verts, iso)
    n = sum(1 for b in b_list if b)
    n_list.append(n)
    d = Dictionary.ByKeyValue("visibility", n)
    v = Topology.SetDictionary(v, d)
    new_verts.append(v)

print(f"Processed {len(new_verts)} valid isovists")
print(f"Visibility counts: {n_list}")
print(f"Min: {min(n_list) if n_list else 'EMPTY'}, Max: {max(n_list) if n_list else 'EMPTY'}")

### //10c Interpolate | Transfer to Graph

In [ ]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=2, key="visibility")

### //10d Derive Vertex Colors

In [ ]:
minValue = min(n_list)
maxValue = max(n_list)
for v in g_verts:
    d = Topology.Dictionary(v)
    vb = Dictionary.ValueAtKey(d, "visibility")
    color = Color.AnyToHex(Color.ByValueInRange(vb, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "vb_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    v = Topology.SetDictionary(v, d)

### //10e Transfer to Shell

In [ ]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
Topology.Show(faces,
              faceColorKey="vb_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)